In [6]:
# libraries for data loading and system/path settings
import json
import sys
import warnings
import os
from pathlib import Path

# libraries for model building and training
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report as sklearn_classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.classification import train_bert
from utils.evaluation import run_testset_stance

# global settings to suppress unproblematic warning messages
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="The sentencepiece tokenizer")

In [7]:
# load the annotated data in json format
with open("../../../01_data/annotations/annotations_augmentations.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][3:]
            sent_dict.add(label)

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

In [8]:
# get all data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into training and test dataset
train_data, val_data = train_test_split(data_with_annotations, test_size=0.2, shuffle=True, random_state=42)

class StanceDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.dataset = []
        for item in data:
            sentence = item["sentence"]
            for ann in item["annotations"]:
                span_text = ann["text"]
                label = label2id[ann["tag"][3:]]
                # combine sentence and target span
                encoded = tokenizer(
                    sentence,
                    span_text,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                )
                self.dataset.append({
                    "input_ids": encoded["input_ids"].squeeze(0),
                    "attention_mask": encoded["attention_mask"].squeeze(0),
                    "label": torch.tensor(label, dtype=torch.long)
                })
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]

In [9]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_names = ["roberta-base", "bert-base-cased", "distilbert-base-cased", "microsoft/deberta-v3-base"]
optimal_configs = {
    "roberta-base": {
        "best_epoch": 8,
        "best_params": {
            "lr": 9e-06,
            "batch_size": 16,
            "weight_decay": 0.01
        }
    },
    "bert-base-cased": {
        "best_epoch": 8,
        "best_params": {
            "lr": 9e-06,
            "batch_size": 16,
            "weight_decay": 0.01
        }
    },
    "distilbert-base-cased": {
        "best_epoch": 8,
        "best_params": {
            "lr": 4e-06,
            "batch_size": 16,
            "weight_decay": 0.3
        }
    },
    "microsoft/deberta-v3-base": {
        "best_epoch": 8,
        "best_params": {
            "lr": 4e-05,
            "batch_size": 16,
            "weight_decay": 0.3
        }
    }
}

In [12]:
# set the device explicitly
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

results = {}

for model_name in model_names:

    # get the hyperparameter configuration
    epochs = optimal_configs[model_name]["best_epoch"]
    lr = optimal_configs[model_name]["best_params"]["lr"]
    batch_size = optimal_configs[model_name]["best_params"]["batch_size"]
    weight_decay = optimal_configs[model_name]["best_params"]["weight_decay"]

    # define the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create tensor dataset and dataloaders
    train_dataset = StanceDataset(train_data, tokenizer, label_to_id, max_len=128)
    val_dataset = StanceDataset(val_data, tokenizer, label_to_id, max_len=128)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # instantiate the model and optimizer
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label_to_id),
        id2label=id_to_label,
        label2id=label_to_id
        ).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # train the model
    train_bert(train_dataloader, model, optimizer, epochs, device, which_task="stance")

    true_labels, pred_labels, _ = run_testset_stance(model=model, test_dataloader=val_dataloader, device=device)
    metrics = sklearn_classification_report(
        [id_to_label[i] for i in true_labels],
        [id_to_label[i] for i in pred_labels],
        output_dict=True
        )
    results[model_name] = {
           "negative_f1": metrics["neg"]["f1-score"],
           "neutral_f1": metrics["neutral"]["f1-score"],
           "positive_f1": metrics["pos"]["f1-score"],
           "macro_f1": metrics["macro avg"]["f1-score"]
           }

Epoch 1/8


Training: 100%|██████████| 171/171 [01:09<00:00,  2.45it/s, loss=0.311]


Average training loss: 0.7586
Epoch 2/8


Training: 100%|██████████| 171/171 [01:09<00:00,  2.46it/s, loss=0.161]


Average training loss: 0.5609
Epoch 3/8


...

Average training loss: 0.1498
Epoch 7/8


Training: 100%|██████████| 171/171 [01:34<00:00,  1.81it/s, loss=0.0064] 


Average training loss: 0.0985
Epoch 8/8


Training: 100%|██████████| 171/171 [01:35<00:00,  1.78it/s, loss=0.00113]


Average training loss: 0.0793


In [17]:
# export the metrics
with open("../eval_results/evaluation_metrics_bert.json", "w") as f:
    json.dump(results, f)